# KUKA KRL Path Visualizer
**Author:** Thejas Dixit Sathyanarayana  
**GitHub:** [Thejas12Dixit](https://github.com/Thejas12Dixit)  

**What this notebook does:**  
Parses a KUKA robot program (`.src` + `.dat` files), visualizes the 3D tool path, displays statistics, and exports a PDF/PNG report.

**KRL = KUKA Robot Language** — the programming language used on KUKA robot controllers.  
- `.src` = the program file: contains motion commands like `PTP`, `LIN`, `CIRC`  
- `.dat` = the data file: contains the actual X,Y,Z coordinates for each named point  

---
Run cells **top to bottom** in order. Each cell builds on the one before it.

## 1. Setup — Load libraries and our parser

In [ ]:
%matplotlib notebook
# %matplotlib notebook = Jupyter magic command
# Tells matplotlib to render plots as interactive widgets INSIDE the notebook
# (instead of opening a separate window or saving to file)
# Must be the very first line before any matplotlib imports

import sys, os
# sys = system tools (used here to modify the Python import path)
# os  = operating system tools (used here to get the current directory)

sys.path.insert(0, os.getcwd())
# sys.path = list of directories Python searches when you do "import something"
# os.getcwd() = "get current working directory" — the folder this notebook is in
# sys.path.insert(0, ...) = add that folder to the FRONT of the search path
# This ensures Python finds krl_parser.py in the same folder as this notebook

from krl_parser import KRLParser, KRLVisualizer
# KRLParser    = reads .src and .dat files and extracts motion data
# KRLVisualizer = takes the parsed data and generates plots / exports

import matplotlib.pyplot as plt
# pyplot = the main matplotlib plotting interface
# We use it here for plt.tight_layout() and plt.show()

print('✔ Modules loaded')  # Confirmation that all imports succeeded

## 2. Parse KRL files
Point the parser at your `.src` file.  
The `.dat` file is **auto-detected** if it's in the same folder with the same name.

**To use your own file:** change `SRC_FILE` to the path of your actual KUKA program.

In [ ]:
# ── Configure file paths ─────────────────────────────────────────────────────
SRC_FILE = 'sample_welding.src'   # Path to the .src file — change this to your file
DAT_FILE = None                   # Path to the .dat file — leave as None for auto-detect
# ─────────────────────────────────────────────────────────────────────────────

parser = KRLParser()
# Create a new parser instance
# KRLParser reads both the .src (motion commands) and .dat (coordinates)

program = parser.parse(SRC_FILE, DAT_FILE)
# parser.parse() does three things:
#   1. Reads the .dat file → extracts all point coordinates (X, Y, Z, A, B, C)
#   2. Reads the .src file → extracts all motion commands (PTP, LIN, CIRC)
#   3. Links each motion command to its coordinates ("resolves" the points)
# Returns a KRLProgram object containing everything

vis = KRLVisualizer(program)
# Create a visualizer from the parsed program
# vis will be used in the cells below for plotting and exporting

# ── Check for warnings ────────────────────────────────────────────────────────
if program.warnings:
    # Warnings occur when a point name in the .src file is not found in the .dat file
    # This can happen if the .dat file is incomplete or the files don't match
    print('⚠ Warnings:')
    for w in program.warnings:
        print(f'  {w}')
else:
    print(f'✔ Parsed successfully — {len(program.motions)} motion commands found')
    # program.motions = list of all parsed motion commands (PTP, LIN, CIRC)

## 3. Program statistics
Shows a summary of the loaded robot program: motion counts, total path distance, workspace envelope (min/max X/Y/Z), and velocity range.

In [ ]:
stats = vis.get_stats()
# get_stats() calculates and returns a dictionary containing:
#   program_name     — name of the .src file (without extension)
#   total_points     — number of points that had valid coordinates
#   ptp_moves        — count of PTP (point-to-point, joint-interpolated) moves
#   lin_moves        — count of LIN (linear Cartesian) moves
#   circ_moves       — count of CIRC (circular arc) moves
#   total_distance   — sum of straight-line distances between consecutive points (mm)
#   x_range          — (min_X, max_X) workspace extent in X direction (mm)
#   y_range          — (min_Y, max_Y) workspace extent in Y direction (mm)
#   z_range          — (min_Z, max_Z) workspace extent in Z direction (mm)
#   min_velocity_mms — slowest LIN/CIRC velocity in mm/s
#   max_velocity_mms — fastest LIN/CIRC velocity in mm/s
#   warnings         — list of any parsing warning messages

print('─' * 40)
print(f"  Program       : {stats['program_name']}")
print(f"  Total points  : {stats['total_points']}")
print(f"  PTP moves     : {stats['ptp_moves']}")    # Joint-space moves (fastest, not Cartesian-linear)
print(f"  LIN moves     : {stats['lin_moves']}")    # Linear Cartesian moves (TCP moves in a straight line)
print(f"  CIRC moves    : {stats['circ_moves']}")   # Circular arc moves (TCP follows an arc through a via-point)
print(f"  Total distance: {stats['total_distance']} mm")  # Approximate path length
print('─' * 40)
print('  Workspace envelope:')           # The 3D bounding box that contains all robot positions
print(f"  X: {stats['x_range'][0]} → {stats['x_range'][1]} mm")  # stats['x_range'] is a tuple (min, max)
print(f"  Y: {stats['y_range'][0]} → {stats['y_range'][1]} mm")
print(f"  Z: {stats['z_range'][0]} → {stats['z_range'][1]} mm")
print('─' * 40)
print(f"  Velocity range: {stats['min_velocity_mms']} → {stats['max_velocity_mms']} mm/s (LIN/CIRC)")
# PTP velocity is in % of max speed so not included here — only LIN/CIRC use m/s

## 4. Interactive 3D path plot
The plot shows the robot's complete tool path in 3D space.

**Color coding:**
- 🔵 Blue lines = PTP moves (joint-interpolated, fastest)
- 🟢 Green lines = LIN moves (linear Cartesian, straight line)
- 🟠 Orange lines = CIRC moves (circular arc)
- 🔴 Red dots = process/weld points
- 🟣 Purple dot = HOME position

**Controls:** Rotate with left-drag · Zoom with scroll · Pan with right-drag

In [ ]:
fig, ax = vis.plot_3d()
# vis.plot_3d() creates a matplotlib 3D figure with the robot path drawn on it
# Returns: fig = the Figure object, ax = the 3D Axes object
# The plot uses matplotlib "Agg" backend internally for drawing,
# but %matplotlib notebook (set in cell 1) makes it interactive in Jupyter

plt.tight_layout()
# tight_layout() automatically adjusts subplot spacing to prevent labels being cut off

plt.show()
# Render the figure in the notebook as an interactive widget
# You can rotate, zoom, and pan the 3D view with the mouse

## 5. Motion sequence table
Shows every motion command in order with its coordinates and velocity.  
Each row = one move the robot makes.  
Colors match the 3D plot: blue=PTP, green=LIN, orange=CIRC.

In [ ]:
import pandas as pd
# pandas = data analysis library. We use it here to create a styled table (DataFrame)
# A DataFrame is like an Excel spreadsheet in Python

rows = []   # Empty list — we'll add one dict per motion command

for i, motion in enumerate(program.motions):
    # program.motions = the ordered list of all parsed motion commands
    # enumerate() gives us both the index i (0, 1, 2...) and the motion object

    if motion.point is None:
        continue   # Skip motions where the point coordinates weren't found in the .dat file

    pt = motion.point   # KRLPoint object with .x, .y, .z, .a, .b, .c attributes

    # Build one row as a dictionary — keys become column headers in the DataFrame
    rows.append({
        '#':          i + 1,                   # Row number (1-based, so +1 since i starts at 0)
        'Type':       motion.motion_type,       # "PTP", "LIN", or "CIRC"
        'Point':      motion.point_name,        # e.g. "P3", "HOME", "P11"
        'X (mm)':     round(pt.x, 1),           # round() to 1 decimal place for clean display
        'Y (mm)':     round(pt.y, 1),
        'Z (mm)':     round(pt.z, 1),
        'A (deg)':    round(pt.a, 1),           # Rotation A (yaw)
        'B (deg)':    round(pt.b, 1),           # Rotation B (pitch)
        'C (deg)':    round(pt.c, 1),           # Rotation C (roll)
        'Velocity':   f"{motion.velocity} {motion.velocity_unit}" if motion.velocity else '—',
        # f-string: formats as e.g. "0.3 m/s" or "80 %". '—' if velocity wasn't parsed
    })

df = pd.DataFrame(rows)
# pd.DataFrame(rows) converts the list of dicts into a table
# Each dict becomes one row; the keys become column names

# ── Style the table ───────────────────────────────────────────────────────────
def color_type(val):
    # This function is applied to each cell in the 'Type' column
    # Returns a CSS style string that sets the background color
    colors = {'PTP': '#D6EAF8', 'LIN': '#D5F5E3', 'CIRC': '#FDEBD0'}
    # Light blue for PTP, light green for LIN, light orange for CIRC
    return f'background-color: {colors.get(val, "white")}'
    # colors.get(val, "white") = look up val in dict; use "white" if not found

df.style.applymap(color_type, subset=['Type'])
# .style = pandas styling interface
# .applymap() = apply a function cell-by-cell
# subset=['Type'] = only apply to the 'Type' column
# The result is displayed as a colored HTML table in Jupyter

## 6. Export to PDF and PNG
Saves the visualization as files you can share or include in documentation.

- **PDF** = 2-page report: page 1 has the 3D path + statistics panel, page 2 has the full motion table
- **PNG** = single image of the 3D path (good for README, LinkedIn posts, reports)

In [ ]:
# ── Export PDF report ────────────────────────────────────────────────────────
pdf_path = vis.export_pdf(f'{program.name}_report.pdf')
# export_pdf() generates a 2-page A4 landscape PDF using matplotlib
# The filename is built from the program name, e.g. "sample_welding_report.pdf"
# Returns the path of the saved file
print(f'✔ PDF saved → {pdf_path}')

# ── Export PNG image ──────────────────────────────────────────────────────────
png_path = vis.export_png(f'{program.name}_path.png')
# export_png() saves a single 3D path image at 150 DPI
# Good resolution for documentation and LinkedIn posts
# Returns the path of the saved file
print(f'✔ PNG saved → {png_path}')

## 7. Try your own file
To use this notebook with a real KUKA program:

1. Copy your `.src` and `.dat` files into the same folder as this notebook
2. Go back to **Cell 2** and change:
   ```python
   SRC_FILE = 'your_program_name.src'
   ```
3. Run all cells again from top to bottom (**Kernel → Restart & Run All**)

**Tip:** If your `.dat` file has a different name or is in a different folder, set:
```python
DAT_FILE = 'path/to/your_program.dat'
```